In [1]:
# Purpose: Import the PySpark DataFrame, ML, evaluation, plotting, and utility components used in this stage.
from pyspark.sql import functions as F
from pyspark.sql.types import (StringType, DoubleType, FloatType, ArrayType, MapType, StructType)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml.functions import vector_to_array

In [2]:
# Purpose: Define the GCS input, output, vector, and model locations used by this stage.
NORMALIZED_SPARK_ROOT = ("gs://federal-contracts/staging/" "federal_contracts_uuid_normalized/v1/")
SILVER_ROOT = ("gs://federal-contracts/silver/" "contracts_ml/v1")
GOLD_ROOT = ("gs://federal-contracts/gold/" "contracts_ml/v1"
)
MODEL_ROOT = (
    "gs://federal-contracts/models/"
    "contracts_ml/v1"
)
PREDICTION_ROOT = (
    "gs://federal-contracts/predictions/"
    "contracts_ml/v1"
)
EDA_ROOT = (
    "gs://federal-contracts/eda/"
    "contracts_ml/v1"
)
QUALITY_ROOT = (
    "gs://federal-contracts/quality/"
    "contracts_ml/v1"
)

In [3]:
# Purpose: Configure Spark shuffle partitioning for the distributed Big Data workload.
spark.conf.set("spark.sql.shuffle.partitions", "192")

In [4]:
# Purpose: Define the GCS input, output, vector, and model locations used by this stage.
UNDER_PROCESSING_PATH = ("gs://federal-contracts/under_processing/contracts_cleaned_v1")
##LAODING THE DATA FROM UNDER_PREPROCESSING
contracts_df = (spark.read.parquet("gs://federal-contracts/under_processing/contracts_cleaned_v1"))

26/08/20 18:10:45 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


In [5]:
# Purpose: Check the dataset width and basic row/column dimensions.
len(contracts_df.columns)


30

In [6]:
# Purpose: Define the project columns used for schema validation, cleaning, feature engineering, or analysis.
EXPECTED_COLUMNS = [
    # -------------------------
    # IDENTIFIERS
    # -------------------------
    "id", "award_id", "piid", "parent_award_piid",
    # -------------------------
    # CONTRACT CHARACTERISTICS
    # -------------------------
    "award_type", "extent_competed", "solicitation_procedures", "type_of_set_aside",
    # -------------------------
    # AGENCIES
    # -------------------------
    "awarding_agency_code", "awarding_agency_name", "awarding_subagency_code",
    "funding_agency_name",
    # -------------------------
    # INDUSTRY
    # -------------------------
    "naics_code",
    "psc_code",
    # -------------------------
    # LOCATION
    # -------------------------
    "recipient_state",
    "recipient_country",
    "pop_state",
    "pop_country",
    # -------------------------
    # RECIPIENT
    # -------------------------
    "recipient_name",
    "recipient_name_normalized",
    "small_business_flag",
    # -------------------------
    # MONEY
    # -------------------------
    "base_and_all_options_value",
    "total_obligation",
    "federal_action_obligation",
    # -------------------------
    # TARGET
    # -------------------------
    "number_of_offers_received",
    # -------------------------
    # DATES
    # -------------------------
    "action_date",
    "period_of_performance_start",
    "period_of_performance_end",
    # -------------------------
    # YEARS
    # -------------------------
    "fiscal_year",
    "source_year"
]

In [7]:
# Purpose: Check the dataset width and basic row/column dimensions.
missing_columns = [c for c in EXPECTED_COLUMNS if c not in contracts_df.columns]
extra_columns = [c for c in contracts_df.columns if c not in EXPECTED_COLUMNS]
print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)
print("Expected:", len(EXPECTED_COLUMNS))
print("Actual:", len(contracts_df.columns))

Missing columns: []
Extra columns: []
Expected: 30
Actual: 30


In [8]:
# Purpose: Restrict the working DataFrame to the expected contract fields for consistent downstream cleaning.
full_clean = contracts_df.select(*EXPECTED_COLUMNS)

## PART B — BEFORE-CLEANING DATA QUALITY REPORT

In [9]:
# Purpose: Define `null_report_fast` to calculate column-level null and empty-value statistics efficiently.
def null_report_fast(df):
    expressions = [F.count("*").alias("__total_rows")]
    for field in df.schema.fields:
        c = field.name
        condition = (F.col(c).isNull())
        #EMPTY STRING AND NULL VALUES 
        if (field.dataType.simpleString()== "string"):
            condition = (condition|(F.trim(F.col(c)) == ""))
        expressions.append(F.sum(F.when(condition,1).otherwise(0)).alias(c))
    result = (df.agg(*expressions).first())
    total = result["__total_rows"]
    rows = []
    for c in df.columns:
        missing = int(result[c] or 0)
        rows.append((c,missing,round(missing/ total* 100,2)))
    return spark.createDataFrame(rows,["column","missing_count","missing_percent"])

In [10]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
null_before = (null_report_fast(full_clean).orderBy(F.desc("missing_percent")))
null_before.show(50, truncate=False)

26/08/20 18:11:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------------------+-------------+---------------+
|column                     |missing_count|missing_percent|
+---------------------------+-------------+---------------+
|type_of_set_aside          |20428142     |76.27          |
|number_of_offers_received  |17846210     |66.63          |
|small_business_flag        |13237903     |49.43          |
|parent_award_piid          |5249136      |19.6           |
|award_type                 |4856585      |18.13          |
|pop_state                  |1785275      |6.67           |
|period_of_performance_end  |971490       |3.63           |
|pop_country                |972029       |3.63           |
|recipient_state            |587265       |2.19           |
|total_obligation           |492269       |1.84           |
|extent_competed            |122677       |0.46           |
|solicitation_procedures    |118674       |0.44           |
|recipient_name             |85321        |0.32           |
|recipient_name_normalized  |85321      

In [11]:
# Purpose: Materialize the Spark DataFrame and verify the number of available records.
original_count = full_clean.count()
print("Rows before cleaning:",f"{original_count:,}")

Rows before cleaning: 26,783,674


## Check exact duplicate rows

In [12]:
# Purpose: Materialize the Spark DataFrame and verify the number of available records.
exact_distinct_count = (full_clean.distinct().count())
print("Exact duplicate rows:",f"{original_count - exact_distinct_count:,}")

Exact duplicate rows: 0


In [13]:
# Purpose: Persist the transformed Spark DataFrame to avoid recomputing expensive distributed transformations.
# from pyspark import StorageLevel
# silver_df.persist(
#     StorageLevel.MEMORY_AND_DISK
# )
# # Materialize
# silver_count = silver_df.count()

## PART C — STRING CLEANING

### -Clean identifier columns

In [14]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Clean ID columns by trimming whitespace and converting empty values to null
ID_COLUMNS = ["id", "award_id", "piid", "parent_award_piid"]
for c in ID_COLUMNS:
    full_clean = full_clean.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), F.lit(None))
        .otherwise(F.trim(F.col(c))))

## ii.Clean categorical/code columns

In [15]:
# Purpose: Prepare `CATEGORY_CODE_COLUMNS` for the next Big Data processing, analysis, or modeling step.
CATEGORY_CODE_COLUMNS = ["award_type", "extent_competed", "solicitation_procedures", "type_of_set_aside",
    "awarding_agency_code", "awarding_subagency_code", "naics_code", "psc_code", "recipient_state",
    "recipient_country",
    "pop_state",
    "pop_country"
]

In [16]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Clean category/code columns by trimming whitespace, converting empty values to null, and standardizing text to uppercase
for c in CATEGORY_CODE_COLUMNS:
    full_clean = full_clean.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), F.lit(None))
        .otherwise(F.upper(F.trim(F.col(c)))))

## iii.Clean agency names

In [17]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Clean agency name columns by trimming whitespace, normalizing spaces, converting empty values to null, and standardizing to uppercase
AGENCY_NAME_COLUMNS = ["awarding_agency_name", "funding_agency_name"]
for c in AGENCY_NAME_COLUMNS:
    full_clean = full_clean.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), F.lit(None))
        .otherwise(F.upper(F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))))

## iv.Clean recipient names

In [18]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Clean recipient names by trimming whitespace, normalizing spaces, converting empty values to null, and uppercasing the normalized name
full_clean = (full_clean.withColumn("recipient_name",
        F.when(F.col("recipient_name").isNull() | (F.trim(F.col("recipient_name")) == ""), F.lit(None))
        .otherwise(F.regexp_replace(F.trim(F.col("recipient_name")), r"\s+", " "))).withColumn(
        "recipient_name_normalized", F.when(
            F.col("recipient_name_normalized").isNull() | (F.trim(F.col("recipient_name_normalized")) == ""),
            F.lit(None)
        ).otherwise(F.upper(F.regexp_replace(F.trim(F.col("recipient_name_normalized")), r"\s+", " ")))))

In [19]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
full_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- award_type: string (nullable = true)
 |-- extent_competed: string (nullable = true)
 |-- solicitation_procedures: string (nullable = true)
 |-- type_of_set_aside: string (nullable = true)
 |-- awarding_agency_code: string (nullable = true)
 |-- awarding_agency_name: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = true)
 |-- funding_agency_name: string (nullable = true)
 |-- naics_code: string (nullable = true)
 |-- psc_code: string (nullable = true)
 |-- recipient_state: string (nullable = true)
 |-- recipient_country: string (nullable = true)
 |-- pop_state: string (nullable = true)
 |-- pop_country: string (nullable = true)
 |-- recipient_name: string (nullable = true)
 |-- recipient_name_normalized: string (nullable = true)
 |-- small_business_flag: boolean (nullable = true)
 |-- base_and_al

In [20]:
# Purpose: Check the dataset width and basic row/column dimensions.
len(full_clean.columns)

30

## PART D — CODE VALIDATION

In [21]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check total rows and count NAICS codes that do not match the expected 2-6 digit numeric format
full_clean.select(F.count("*").alias("total"), F.sum(F.when(
            F.col("naics_code").isNotNull() & (~F.col("naics_code").rlike(r"^[0-9]{2,6}$")), 1).otherwise(0)
    ).alias("invalid_naics_format")).show()

+--------+--------------------+
|   total|invalid_naics_format|
+--------+--------------------+
|26783674|                   0|
+--------+--------------------+



In [29]:
# Purpose: Execute the code for the notebook section: PART D — CODE VALIDATION.
## removing the invalid codes

## Inspect PSC

In [23]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check total rows and count PSC codes containing invalid characters
full_clean.select(F.count("*").alias("total"), F.sum(F.when(
            F.col("psc_code").isNotNull() & (~F.col("psc_code").rlike(r"^[A-Z0-9]+$")), 1).otherwise(0)
    ).alias("invalid_psc_format")).show()

+--------+------------------+
|   total|invalid_psc_format|
+--------+------------------+
|26783674|                 0|
+--------+------------------+



## PART E — DATE CLEANING

In [24]:
# Purpose: Define the project columns used for schema validation, cleaning, feature engineering, or analysis.
# Check each date column for unrealistic years outside the 2000-2100 range
DATE_COLUMNS = ["action_date", "period_of_performance_start", "period_of_performance_end"]
for c in DATE_COLUMNS:
    print("\n", c)
    (full_clean.filter(F.col(c).isNotNull()).withColumn("year", F.year(F.col(c)))
        .filter((F.col("year") < 2000) | (F.col("year") > 2100)).groupBy("year").count().orderBy("year")
        .show(100))


 action_date


+----+-----+
|year|count|
+----+-----+
+----+-----+


 period_of_performance_start


+----+-----+
|year|count|
+----+-----+
+----+-----+


 period_of_performance_end


+----+-----+
|year|count|
+----+-----+
|1996|    1|
|1997|    1|
|1998|    1|
|1999|    4|
+----+-----+



## Null impossible dates

In [27]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
for c in DATE_COLUMNS:
    full_clean = (full_clean.withColumn(c, F.when(F.col(c).isNull(), F.lit(None).cast("date")).when(
                F.year(F.col(c)).between(
                    2000,
                    2100
                ),
                F.col(c)
            )
            .otherwise(
                F.lit(None).cast("date")
            )
        )
    )

## PART F — YEAR VALIDATION

## Cast years correctly

In [28]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
full_clean = (full_clean.withColumn("fiscal_year", F.col("fiscal_year").cast("int")).withColumn(
        "source_year",
        F.col(
            "source_year"
        ).cast("int")
    )
)

## Validate source year

In [29]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
full_clean = (full_clean.withColumn("source_year", F.when(F.col("source_year").between(2009, 2024),
            F.col("source_year")
        )
    )
)

In [30]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
full_clean = (full_clean.withColumn("fiscal_year", F.when(F.col("fiscal_year").between(2000, 2100),
            F.col("fiscal_year")
        )
    )
)

## Check fiscal_year vs source_year

In [31]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Compare fiscal year and source year values to identify matches, differences, and nulls
year_comparison = (full_clean.select(F.count("*").alias("total_rows"),
        F.sum(F.when(F.col("fiscal_year") == F.col("source_year"), 1).otherwise(0)).alias("same_year"),
        F.sum(F.when(F.col("fiscal_year") != F.col("source_year"), 1).otherwise(0)).alias("different_year"),
        F.sum(F.when(F.col("fiscal_year").isNull() | F.col("source_year").isNull(), 1).otherwise(0)).alias("null_year")
    ))
year_comparison.show()

+----------+---------+--------------+---------+
|total_rows|same_year|different_year|null_year|
+----------+---------+--------------+---------+
|  26783674| 21082036|       5701638|        0|
+----------+---------+--------------+---------+



In [32]:
# Purpose: Aggregate contract records by year to inspect temporal coverage and target availability.
# Show the 50 most common fiscal year and source year mismatches
(full_clean.filter(F.col("fiscal_year").isNotNull() & F.col("source_year").isNotNull()
        & (F.col("fiscal_year") != F.col("source_year"))).groupBy("fiscal_year", "source_year").count()
    .orderBy(F.desc("count")).show(50)
)

+-----------+-----------+-------+
|fiscal_year|source_year|  count|
+-----------+-----------+-------+
|       2024|       2023|1471546|
|       2023|       2022|1383308|
|       2021|       2020|1369947|
|       2020|       2019|1332576|
|       2010|       2009|  31688|
|       2019|       2018|  13928|
|       2018|       2017|  13702|
|       2011|       2010|  12810|
|       2015|       2014|  12799|
|       2016|       2015|  12443|
|       2012|       2011|  12159|
|       2017|       2016|  11961|
|       2013|       2012|  11768|
|       2014|       2013|  11003|
+-----------+-----------+-------+



## TARGET CLEANING

## Clean number_of_offers_received

In [35]:
# Purpose: Aggregate the Spark data to produce summary statistics for this analysis.
full_clean.groupBy('number_of_offers_received').count().orderBy(F.col('number_of_offers_received')).show(10)

+-------------------------+--------+
|number_of_offers_received|   count|
+-------------------------+--------+
|                     NULL|17846210|
|                        0|   32695|
|                        1| 3508424|
|                        2| 1020884|
|                        3|  987371|
|                        4|  520073|
|                        5|  544531|
|                        6|  222695|
|                        7|  151523|
|                        8|  148097|
+-------------------------+--------+
only showing top 10 rows



In [ ]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Keep valid non-negative offer counts as integers and convert invalid negative values to null
full_clean = full_clean.withColumn("number_of_offers_received", F.when(F.col("number_of_offers_received") >= 0,
        F.col("number_of_offers_received").cast("int")).otherwise(F.lit(None).cast("int")))

In [ ]:
# Purpose: Aggregate the Spark data to produce summary statistics for this analysis.
full_clean.groupBy('number_of_offers_received').count().orderBy(F.col('count').desc()).show(10)

## Inspect target distribution

In [41]:
# Purpose: Calculate robust distribution statistics and approximate percentiles for the selected numeric field.
# Show key percentiles and the maximum value for the number of offers received
#50% of records have this many offers or fewer
full_clean.select(F.expr("""
        percentile_approx(
            number_of_offers_received,
            array(0.5, 0.75, 0.90, 0.95, 0.99, 0.999) 
            
        )
    """).alias("offers_percentiles"), F.max("number_of_offers_received").alias("max_offers")
).show(truncate=False)

+-------------------------+----------+
|offers_percentiles       |max_offers|
+-------------------------+----------+
|[2, 6, 22, 999, 999, 999]|999       |
+-------------------------+----------+



## MONETARY CLEANING

In [42]:
# Purpose: Define the project columns used for schema validation, cleaning, feature engineering, or analysis.
MONEY_COLUMNS = ["base_and_all_options_value", "total_obligation", "federal_action_obligation"]

In [43]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Convert all money-related columns to double data type
for c in MONEY_COLUMNS:
    full_clean = full_clean.withColumn(c, F.col(c).cast("double"))

### Check suspicious extreme monetary values

In [40]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check total rows and count sentinel-like extreme values in each money column
full_clean.select(F.count("*").alias("total"), *[
        F.sum(F.when(F.abs(F.col(c)) >= 9e11, 1).otherwise(0)).alias(f"{c}_sentinel_like")
        for c in MONEY_COLUMNS]).show(truncate=False)

+--------+----------------------------------------+------------------------------+---------------------------------------+
|total   |base_and_all_options_value_sentinel_like|total_obligation_sentinel_like|federal_action_obligation_sentinel_like|
+--------+----------------------------------------+------------------------------+---------------------------------------+
|26783674|1470                                    |0                             |0                                      |
+--------+----------------------------------------+------------------------------+---------------------------------------+



## Remove only sentinel-like extremes

In [44]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Replace NaN and sentinel-like extreme values in money columns with null
MONEY_SENTINEL_THRESHOLD = 9e11
for c in MONEY_COLUMNS:
    full_clean = full_clean.withColumn(c, F.when(F.isnan(F.col(c)), F.lit(None).cast("double"))
        .when(F.abs(F.col(c)) >= MONEY_SENTINEL_THRESHOLD, F.lit(None).cast("double")).otherwise(F.col(c)))

## PART I — BOOLEAN

In [45]:
# Purpose: Normalize the small-business indicator into a numeric feature.
# Convert the small business boolean flag into a numeric 1.0/0.0 feature while keeping missing values as null
full_clean = full_clean.withColumn("small_business_flag_i", F.when(F.col("small_business_flag") == True, 1.0)
    .when(F.col("small_business_flag") == False, 0.0).otherwise(F.lit(None).cast("double")))

## DEDUPLICATION

In [46]:
# Purpose: Aggregate the Spark data to produce summary statistics for this analysis.
# Find duplicate non-null contract IDs and show the first 30 duplicate IDs
duplicate_ids = (full_clean.filter(F.col("id").isNotNull()).groupBy("id").count().filter(F.col("count") > 1))
duplicate_ids.show(30, truncate=False)

+---+-----+
|id |count|
+---+-----+
+---+-----+



## FEATURE ENGINEERING

In [47]:
# Purpose: Engineer compact NAICS and PSC category features for analysis and modeling.
# Create a 2-digit NAICS sector feature from the NAICS code
full_features = full_clean.withColumn("naics2",
    F.when(F.length(F.col("naics_code")) >= 2, F.substring(F.col("naics_code"), 1, 2)))

## PSC group

In [48]:
# Purpose: Engineer compact NAICS and PSC category features for analysis and modeling.
# Create a 2-character PSC group feature from the PSC code
full_features = full_features.withColumn("psc2",
    F.when(F.length(F.col("psc_code")) >= 2, F.substring(F.col("psc_code"), 1, 2)))

In [49]:
# Purpose: Execute the code for the notebook section: PSC group.
## Action-date features

In [50]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Create year, month, and quarter features from the contract action date
full_features = (full_features.withColumn("action_year", F.year(F.col("action_date")).cast("double"))
    .withColumn("action_month", F.month(F.col("action_date")).cast("double"))
    .withColumn("action_quarter", F.quarter(F.col("action_date")).cast("double")))

In [52]:
# Purpose: Engineer and validate contract duration as a derived numeric feature.
# Create contract duration in days from the performance start and end dates
full_features = full_features.withColumn("duration_days",
    F.datediff(F.col("period_of_performance_end"), F.col("period_of_performance_start")).cast("double"))

In [53]:
# Purpose: Engineer and validate contract duration as a derived numeric feature.
# Keep realistic contract durations between 0 and 4,650 days and convert other values to null
full_features = full_features.withColumn("duration_days",
    F.when((F.col("duration_days") >= 0) & (F.col("duration_days") <= 4650), F.col("duration_days"))
    .otherwise(F.lit(None).cast("double")))

In [54]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Create a binary indicator showing whether the contract has a parent award
full_features = full_features.withColumn("has_parent_award",
    F.when(F.col("parent_award_piid").isNotNull(), 1.0).otherwise(0.0))

In [59]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
len(full_features.columns)
full_features.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- award_type: string (nullable = true)
 |-- extent_competed: string (nullable = true)
 |-- solicitation_procedures: string (nullable = true)
 |-- type_of_set_aside: string (nullable = true)
 |-- awarding_agency_code: string (nullable = true)
 |-- awarding_agency_name: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = true)
 |-- funding_agency_name: string (nullable = true)
 |-- naics_code: string (nullable = true)
 |-- psc_code: string (nullable = true)
 |-- recipient_state: string (nullable = true)
 |-- recipient_country: string (nullable = true)
 |-- pop_state: string (nullable = true)
 |-- pop_country: string (nullable = true)
 |-- recipient_name: string (nullable = true)
 |-- recipient_name_normalized: string (nullable = true)
 |-- small_business_flag: boolean (nullable = true)
 |-- base_and_al

# FINAL CATEGORICAL FEATURES

In [63]:
# Purpose: Define the project columns used for schema validation, cleaning, feature engineering, or analysis.
# Final categorical features used for analysis and machine learning
FINAL_CATEGORICAL = ["award_type", "extent_competed", "solicitation_procedures", "type_of_set_aside",
    "awarding_agency_code", "awarding_subagency_code", "funding_agency_name",
    "naics2", "psc2", "recipient_state", "recipient_country", "pop_state", "pop_country"]

In [64]:
# Purpose: Execute the code for the notebook section: FINAL CATEGORICAL FEATURES.
## Categorical NULL handling

In [62]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
# Replace missing values in final categorical features with "__UNKNOWN__"
for c in FINAL_CATEGORICAL:
    full_features = full_features.withColumn(c, F.coalesce(F.col(c), F.lit("__UNKNOWN__")))

## CREATE TARGETS

In [65]:
# Purpose: Create or clean the machine-learning target column used for model training.
# Create the machine learning target label from the number of offers received
full_features = full_features.withColumn("offers_label",
    F.when(F.col("number_of_offers_received").isNotNull(), F.col("number_of_offers_received").cast("double")))

In [66]:
# Purpose: Create or clean the machine-learning target column used for model training.
# Create the machine learning target label from positive contract award values
full_features = full_features.withColumn("award_value_label",
    F.when(F.col("base_and_all_options_value") > 0, F.col("base_and_all_options_value").cast("double")))

In [68]:
# Purpose: Check the dataset width and basic row/column dimensions.
len(full_features.columns)

40

In [69]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
full_features.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- award_type: string (nullable = false)
 |-- extent_competed: string (nullable = false)
 |-- solicitation_procedures: string (nullable = false)
 |-- type_of_set_aside: string (nullable = false)
 |-- awarding_agency_code: string (nullable = false)
 |-- awarding_agency_name: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = false)
 |-- funding_agency_name: string (nullable = false)
 |-- naics_code: string (nullable = true)
 |-- psc_code: string (nullable = true)
 |-- recipient_state: string (nullable = false)
 |-- recipient_country: string (nullable = false)
 |-- pop_state: string (nullable = false)
 |-- pop_country: string (nullable = false)
 |-- recipient_name: string (nullable = true)
 |-- recipient_name_normalized: string (nullable = true)
 |-- small_business_flag: boolean (nullable = true)
 |-- 

In [72]:
# Purpose: Prepare `silver_clean` for the next Big Data processing, analysis, or modeling step.
silver_clean = full_features

In [73]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
silver_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- award_type: string (nullable = false)
 |-- extent_competed: string (nullable = false)
 |-- solicitation_procedures: string (nullable = false)
 |-- type_of_set_aside: string (nullable = false)
 |-- awarding_agency_code: string (nullable = false)
 |-- awarding_agency_name: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = false)
 |-- funding_agency_name: string (nullable = false)
 |-- naics_code: string (nullable = true)
 |-- psc_code: string (nullable = true)
 |-- recipient_state: string (nullable = false)
 |-- recipient_country: string (nullable = false)
 |-- pop_state: string (nullable = false)
 |-- pop_country: string (nullable = false)
 |-- recipient_name: string (nullable = true)
 |-- recipient_name_normalized: string (nullable = true)
 |-- small_business_flag: boolean (nullable = true)
 |-- 

In [73]:
# Purpose: Define the GCS input, output, vector, and model locations used by this stage.
SILVER_PATH = "gs://federal-contracts/silver/contracts_cleaned/"
# silver_clean.write.mode('overwrite').partitionBy('fiscal_year').parquet(SILVER_PATH)

# CREATE THE ML-READY TABLE

In [74]:
# Purpose: Load the required Parquet data into `ml_df` for this Big Data processing stage.
ml_df=spark.read.parquet(SILVER_PATH)

In [75]:
# Purpose: Prepare `ML_READY_COLUMNS` for the next Big Data processing, analysis, or modeling step.
ML_READY_COLUMNS = [
    # ========================
    # METADATA / IDENTIFIERS
    # ========================
    "id", "award_id", "piid", "parent_award_piid",
    # ========================
    # TIME / SPLIT METADATA
    # ========================
    "source_year", "fiscal_year",
    # ========================
    # CATEGORICAL ML FEATURES
    # ========================
    "award_type", "extent_competed", "solicitation_procedures", "type_of_set_aside", "awarding_agency_code",
    "awarding_subagency_code",
    "funding_agency_name",
    "naics2",
    "psc2",
    "recipient_state",
    "recipient_country",
    "pop_state",
    "pop_country",
    # ========================
    # NUMERIC ML FEATURES
    # ========================
    "action_year",
    "action_month",
    "action_quarter",
    "duration_days",
    "has_parent_award",
    "small_business_flag_i",
    # ========================
    # ORIGINAL MONEY
    # ========================
    "base_and_all_options_value",
    "total_obligation",
    "federal_action_obligation",
    # ========================
    # ORIGINAL OFFERS TARGET
    # ========================
    "number_of_offers_received",
    # ========================
    # ML TARGETS
    # ========================
    "offers_label",
    "award_value_label"
]

In [76]:
# Purpose: Select the columns required for the next Big Data processing or evaluation step.
ml_ready = (ml_df.select(*ML_READY_COLUMNS))

In [77]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
ml_ready.show(10,truncate=False)

+------------------------------------+------------------------------------------------+---------------+-----------------+-----------+-----------+-----------+--------------------------------------------------+------------------------------------------+------------------+--------------------+-----------------------+-------------------------------+------+----+---------------+-----------------+-----------+-----------+-----------+------------+--------------+-------------+----------------+---------------------+--------------------------+----------------+-------------------------+-------------------------+------------+-----------------+
|id                                  |award_id                                        |piid           |parent_award_piid|source_year|fiscal_year|award_type |extent_competed                                   |solicitation_procedures                   |type_of_set_aside |awarding_agency_code|awarding_subagency_code|funding_agency_name            |naics2|psc2|rec

# FINAL CLEANING VALIDATION

In [78]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check missing-value percentages after cleaning and show the columns with the highest missingness
null_after = null_report_fast(ml_ready).orderBy(F.desc("missing_percent"))
null_after.show(50, truncate=False)

+--------------------------+-------------+---------------+
|column                    |missing_count|missing_percent|
+--------------------------+-------------+---------------+
|number_of_offers_received |17846210     |66.63          |
|offers_label              |17846210     |66.63          |
|small_business_flag_i     |13237903     |49.43          |
|parent_award_piid         |5249136      |19.6           |
|award_value_label         |4064825      |15.18          |
|duration_days             |1060925      |3.96           |
|total_obligation          |492269       |1.84           |
|base_and_all_options_value|1470         |0.01           |
|awarding_agency_code      |0            |0.0            |
|fiscal_year               |0            |0.0            |
|awarding_subagency_code   |0            |0.0            |
|federal_action_obligation |0            |0.0            |
|funding_agency_name       |0            |0.0            |
|recipient_state           |0            |0.0           

# Validate target cleaning

In [ ]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check missing-value percentages after cleaning and show the columns with the highest missingness
null_after = null_report_fast(ml_ready).orderBy(F.desc("missing_percent"))
null_after.show(50, truncate=False)

In [79]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check negative offer values and count valid machine learning targets
ml_ready.select(F.sum(F.when(F.col("number_of_offers_received") < 0, 1).otherwise(0)).alias("negative_offers"),
    F.count("offers_label").alias("valid_offers_targets"),
    F.count("award_value_label").alias("valid_value_targets")).show()

+---------------+--------------------+-------------------+
|negative_offers|valid_offers_targets|valid_value_targets|
+---------------+--------------------+-------------------+
|              0|             8937464|           22718849|
+---------------+--------------------+-------------------+



## Validate money sentinels

In [80]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check whether any sentinel-like extreme values remain in the money columns after cleaning
ml_ready.select(*[F.sum(F.when(F.abs(F.col(c)) >= 9e11, 1).otherwise(0)).alias(f"{c}_remaining_sentinels")
        for c in MONEY_COLUMNS]).show(truncate=False)

+----------------------------------------------+------------------------------------+---------------------------------------------+
|base_and_all_options_value_remaining_sentinels|total_obligation_remaining_sentinels|federal_action_obligation_remaining_sentinels|
+----------------------------------------------+------------------------------------+---------------------------------------------+
|0                                             |0                                   |0                                            |
+----------------------------------------------+------------------------------------+---------------------------------------------+



## Validate duration

In [81]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Check for negative contract durations and durations longer than 10 years
ml_ready.select(F.sum(F.when(F.col("duration_days") < 0, 1).otherwise(0)).alias("negative_duration"),
    F.sum(F.when(F.col("duration_days") > 3650, 1).otherwise(0)).alias("duration_over_10_years")).show()

+-----------------+----------------------+
|negative_duration|duration_over_10_years|
+-----------------+----------------------+
|                0|                 28297|
+-----------------+----------------------+



# Check categorical cardinality

In [82]:
# Purpose: Display the resulting Spark rows or summary table for validation and interpretation.
# Count the number of distinct values in each final categorical feature
cardinality = ml_ready.agg(*[F.countDistinct(F.col(c)).alias(c) for c in FINAL_CATEGORICAL])
cardinality.show(truncate=False)

+----------+---------------+-----------------------+-----------------+--------------------+-----------------------+-------------------+------+----+---------------+-----------------+---------+-----------+
|award_type|extent_competed|solicitation_procedures|type_of_set_aside|awarding_agency_code|awarding_subagency_code|funding_agency_name|naics2|psc2|recipient_state|recipient_country|pop_state|pop_country|
+----------+---------------+-----------------------+-----------------+--------------------+-----------------------+-------------------+------+----+---------------+-----------------+---------+-----------+
|10        |11             |11                     |27               |73                  |180                    |101                |25    |177 |62             |303              |60       |446        |
+----------+---------------+-----------------------+-----------------+--------------------+-----------------------+-------------------+------+----+---------------+-----------------+---

In [87]:
# Purpose: Execute the code for the notebook section: Check categorical cardinality.
cardinality.toPandas()

,award_type,extent_competed,solicitation_procedures,type_of_set_aside,awarding_agency_code,awarding_subagency_code,funding_agency_name,naics2,psc2,recipient_state,recipient_country,pop_state,pop_country
0,10,11,11,27,73,180,101,25,177,62,303,60,446


In [91]:
# Purpose: Apply the required Spark column transformation for cleaning or feature engineering.
zero_offers_pct = (full_clean.filter(F.col("number_of_offers_received").isNotNull()).agg(
        F.count("*").alias("valid_offers"), F.sum(F.when(F.col("number_of_offers_received") == 0,
                1
            ).otherwise(0)
        ).alias("zero_offers")
    )
    .withColumn(
        "zero_offers_pct",
        F.round(
            F.col("zero_offers")
            / F.col("valid_offers")
            * 100,
            2
        )
    )
)
zero_offers_pct.show()

+------------+-----------+---------------+
|valid_offers|zero_offers|zero_offers_pct|
+------------+-----------+---------------+
|     8937464|      32695|           0.37|
+------------+-----------+---------------+



In [87]:
# Purpose: Define the GCS input, output, vector, and model locations used by this stage.
SILVER_ROOT = ("gs://federal-contracts/" "silver/contracts_ml/v1")
GOLD_ROOT = ("gs://federal-contracts/" "gold/contracts_ml/v1")
MODEL_ROOT = ("gs://federal-contracts/" "models/contracts_ml/v1"
)

In [88]:
# Purpose: Define `save_stage` to encapsulate the `save_stage` processing logic for reuse.
# Save a DataFrame as partitioned Parquet and optionally as CSV, converting complex columns to JSON for CSV compatibility
def save_stage(df, root, stage_name, partition_col="source_year", save_csv=False):
    parquet_path = f"{root}/{stage_name}/parquet"
    writer = df.write.mode("overwrite")
    if partition_col is not None and partition_col in df.columns:
        writer = writer.partitionBy(partition_col)
    writer.parquet(parquet_path)
    print("Parquet saved:", parquet_path)
    if save_csv:
        csv_path = f"{root}/{stage_name}/csv"
        csv_df = df
        for field in csv_df.schema.fields:
            if isinstance(field.dataType, (ArrayType, MapType, StructType)):
                csv_df = csv_df.withColumn(field.name, F.to_json(F.col(field.name)))
        csv_writer = (csv_df.write.mode("overwrite").option("header", True).option("nullValue", ""))
        if partition_col is not None and partition_col in csv_df.columns:
            csv_writer = csv_writer.partitionBy(partition_col)
        csv_writer.csv(csv_path)
        print("CSV saved:", csv_path)

In [89]:
# Purpose: Execute the code for the notebook section: Check categorical cardinality.
save_stage(silver_clean, SILVER_ROOT, "03_clean", save_csv=False)

Parquet saved: gs://federal-contracts/silver/contracts_ml/v1/03_clean/parquet


In [90]:
# Purpose: Execute the code for the notebook section: Check categorical cardinality.
save_stage(ml_ready, SILVER_ROOT, "04_ml_ready", save_csv=False)

Parquet saved: gs://federal-contracts/silver/contracts_ml/v1/04_ml_ready/parquet


In [93]:
# Purpose: Check the dataset width and basic row/column dimensions.
len(ml_ready.columns)

31

# TEMPORAL TRAIN / VALIDATION / TEST

In [89]:
# Purpose: Create the time-based train, validation, and test split from `source_year`.
# Create time-based train, validation, and test splits using source year
ml_split = ml_ready.withColumn("split", F.when(F.col("source_year") <= 2022, F.lit("train"))
    .when(F.col("source_year") == 2023, F.lit("validation")).when(F.col("source_year") == 2024, F.lit("test")))

In [90]:
# Purpose: Filter the Spark records to the valid subset required for the next analysis or modeling step.
ml_split = (ml_split.filter(F.col("split").isNotNull()))

In [94]:
# Purpose: Aggregate contract records by year to inspect temporal coverage and target availability.
# Validate split
(ml_split.groupBy("split", "source_year").count().orderBy("source_year"
    )
    .show()
)

26/08/18 22:53:48 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.EncodeStringCaseWhenInAggregate effective 1 times.



+----------+-----------+-------+
|     split|source_year|  count|
+----------+-----------+-------+
|     train|       2009|  31688|
|     train|       2010| 167857|
|     train|       2011|  75313|
|     train|       2012|  68367|
|     train|       2013|  63733|
|     train|       2014|  73080|
|     train|       2015|  68165|
|     train|       2016|  72640|
|     train|       2017|  80728|
|     train|       2018|  76422|
|     train|       2019|1381001|
|     train|       2020|6297347|
|     train|       2021|5000361|
|     train|       2022|1383308|
|validation|       2023|6751368|
|      test|       2024|5192296|
+----------+-----------+-------+



In [92]:
# Purpose: Aggregate the Spark data to produce summary statistics for this analysis.
# Count total records and valid target records in each train, validation, and test split
(ml_split.groupBy("split").agg(F.count("*").alias("records"),
        F.count("offers_label").alias("offers_target_records"),
        F.count("award_value_label").alias("award_value_target_records")).show())

26/08/20 19:47:53 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.EncodeStringCaseWhenInAggregate effective 1 times.



+----------+--------+---------------------+--------------------------+
|     split| records|offers_target_records|award_value_target_records|
+----------+--------+---------------------+--------------------------+
|     train|14840010|              5128990|                  12543441|
|      test| 5192296|              1656042|                   4420560|
|validation| 6751368|              2152432|                   5754848|
+----------+--------+---------------------+--------------------------+



In [97]:
# Purpose: Execute the code for the notebook section: TEMPORAL TRAIN / VALIDATION / TEST.
# savingthe split

In [96]:
# Purpose: Filter the Spark records to the valid subset required for the next analysis or modeling step.
for split_name in ["train", "validation", "test"]:
    split_df = (ml_split.filter(F.col("split") == split_name
        )
    )
    save_stage(
        split_df,
        f"{SILVER_ROOT}/splits",
        split_name,
        partition_col=None,
        save_csv=False
    )

Parquet saved: gs://federal-contracts/silver/contracts_ml/v1/splits/train/parquet


Parquet saved: gs://federal-contracts/silver/contracts_ml/v1/splits/validation/parquet


Parquet saved: gs://federal-contracts/silver/contracts_ml/v1/splits/test/parquet


## UPDATED SPARK ML FEATURES

### Final categorical features

In [96]:
# Purpose: Prepare `CATEGORICAL_FEATURES` for the next Big Data processing, analysis, or modeling step.
CATEGORICAL_FEATURES = ["award_type", "extent_competed", "solicitation_procedures", "type_of_set_aside",
    "awarding_agency_code", "awarding_subagency_code", "funding_agency_name", "naics2", "psc2",
    "recipient_state",
    "recipient_country",
    "pop_state",
    "pop_country"
]

In [95]:
# Purpose: Execute the code for the notebook section: Final categorical features.
# Updated numerical features
NUMERIC_FEATURES = ["fiscal_year", "action_year", "action_month", "action_quarter", "duration_days",
    "has_parent_award", "small_business_flag_i"]

# STRING INDEXER + OHE + TRAIN-ONLY IMPUTATION

In [97]:
# Purpose: Execute the code for the notebook section: STRING INDEXER + OHE + TRAIN-ONLY IMPUTATION.
# Build output column names for indexed, encoded, and imputed ML features
indexed_columns = [f"{c}_idx" for c in CATEGORICAL_FEATURES]
encoded_columns = [f"{c}_ohe" for c in CATEGORICAL_FEATURES]
imputed_columns = [f"{c}_imp" for c in NUMERIC_FEATURES]

In [113]:
# Purpose: Build the Spark ML feature-preparation pipeline for categorical and numeric model inputs.
# string indexer
indexer = StringIndexer(inputCols=CATEGORICAL_FEATURES, outputCols=indexed_columns, handleInvalid="keep")

In [112]:
# Purpose: Build the Spark ML feature-preparation pipeline for categorical and numeric model inputs.
# One-hot encoding
encoder = OneHotEncoder(inputCols=indexed_columns, outputCols=encoded_columns, handleInvalid="keep")

In [111]:
# Purpose: Execute the code for the notebook section: STRING INDEXER + OHE + TRAIN-ONLY IMPUTATION.
# Median imputation
imputer = Imputer(strategy="median", inputCols=NUMERIC_FEATURES, outputCols=imputed_columns)

In [110]:
# Purpose: Build the Spark ML feature-preparation pipeline for categorical and numeric model inputs.
# VectorAssembler
assembler = VectorAssembler(inputCols=(encoded_columns + imputed_columns), outputCol="features_raw",
    handleInvalid="keep")

In [109]:
# Purpose: Build the Spark ML feature-preparation pipeline for categorical and numeric model inputs.
# StandardScaler
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)

In [114]:
# Purpose: Execute the code for the notebook section: STRING INDEXER + OHE + TRAIN-ONLY IMPUTATION.
# Complete feature pipeline
feature_pipeline = Pipeline(stages=[indexer, encoder, imputer, assembler, scaler])

# OFFERS ML DATA (offers col)

In [102]:
# Purpose: Create or clean the machine-learning target column used for model training.
# Keep valid offer targets and create a log-transformed label for model training
offers_data = (ml_split.filter(F.col("offers_label").isNotNull()).filter(F.col("offers_label") >= 0)
    .withColumn("label", F.log1p(F.col("offers_label"))))
#because number_of_offers_received is highly right-skewed.

In [103]:
# Purpose: Filter the Spark records to the valid subset required for the next analysis or modeling step.
# Separate the offers data into temporal training, validation, and test datasets
offers_train = offers_data.filter(F.col("split") == "train")
offers_val = offers_data.filter(F.col("split") == "validation")
offers_test = offers_data.filter(F.col("split") == "test")

In [104]:
# Purpose: Materialize the Spark DataFrame and verify the number of available records.
print("Train:", f"{offers_train.count():,}")
print("Validation:", f"{offers_val.count():,}")
print("Test:", f"{offers_test.count():,}"
)

Train: 5,128,990


Validation: 2,152,432


Test: 1,656,042


In [105]:
# Purpose: Execute the code for the notebook section: OFFERS ML DATA (offers col).
# Fit preprocessing on train data

In [106]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
offers_train.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- award_type: string (nullable = true)
 |-- extent_competed: string (nullable = true)
 |-- solicitation_procedures: string (nullable = true)
 |-- type_of_set_aside: string (nullable = true)
 |-- awarding_agency_code: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = true)
 |-- funding_agency_name: string (nullable = true)
 |-- naics2: string (nullable = true)
 |-- psc2: string (nullable = true)
 |-- recipient_state: string (nullable = true)
 |-- recipient_country: string (nullable = true)
 |-- pop_state: string (nullable = true)
 |-- pop_country: string (nullable = true)
 |-- action_year: double (nullable = true)
 |-- action_month: double (nullable = true)
 |-- action_quarter: double (nullable = true)
 |-- durat

In [ ]:
# Purpose: Execute the code for the notebook section: OFFERS ML DATA (offers col).
#fitting the offers data 

In [115]:
# Purpose: Fit the configured Spark ML pipeline or regression model on the training data.
offers_feature_model = (feature_pipeline.fit(offers_train))

In [119]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
offers_train.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- award_type: string (nullable = true)
 |-- extent_competed: string (nullable = true)
 |-- solicitation_procedures: string (nullable = true)
 |-- type_of_set_aside: string (nullable = true)
 |-- awarding_agency_code: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = true)
 |-- funding_agency_name: string (nullable = true)
 |-- naics2: string (nullable = true)
 |-- psc2: string (nullable = true)
 |-- recipient_state: string (nullable = true)
 |-- recipient_country: string (nullable = true)
 |-- pop_state: string (nullable = true)
 |-- pop_country: string (nullable = true)
 |-- action_year: double (nullable = true)
 |-- action_month: double (nullable = true)
 |-- action_quarter: double (nullable = true)
 |-- durat

In [120]:
# Purpose: Execute the code for the notebook section: OFFERS ML DATA (offers col).
# Apply the fitted feature engineering pipeline to the training, validation, and test datasets
offers_train_f = offers_feature_model.transform(offers_train)
offers_val_f = offers_feature_model.transform(offers_val)
offers_test_f = offers_feature_model.transform(offers_test)

In [122]:
# Purpose: Check the dataset width and basic row/column dimensions.
len(offers_train_f.columns)

68

In [123]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
offers_train_f.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- award_type: string (nullable = true)
 |-- extent_competed: string (nullable = true)
 |-- solicitation_procedures: string (nullable = true)
 |-- type_of_set_aside: string (nullable = true)
 |-- awarding_agency_code: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = true)
 |-- funding_agency_name: string (nullable = true)
 |-- naics2: string (nullable = true)
 |-- psc2: string (nullable = true)
 |-- recipient_state: string (nullable = true)
 |-- recipient_country: string (nullable = true)
 |-- pop_state: string (nullable = true)
 |-- pop_country: string (nullable = true)
 |-- action_year: double (nullable = true)
 |-- action_month: double (nullable = true)
 |-- action_quarter: double (nullable = true)
 |-- durat

In [125]:
# Purpose: Execute the code for the notebook section: OFFERS ML DATA (offers col).
# Final offers vectors
VECTOR_COLUMNS_OFFERS = ["id", "award_id", "piid", "source_year", "split",
    "offers_label", "label", "features"]

In [126]:
# Purpose: Select the columns required for the next Big Data processing or evaluation step.
offers_train_vector = (offers_train_f.select(*VECTOR_COLUMNS_OFFERS))
offers_val_vector = (offers_val_f.select(*VECTOR_COLUMNS_OFFERS
    )
)
offers_test_vector = (
    offers_test_f
    .select(
        *VECTOR_COLUMNS_OFFERS
    )
)

In [127]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
offers_test_f.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- parent_award_piid: string (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- award_type: string (nullable = true)
 |-- extent_competed: string (nullable = true)
 |-- solicitation_procedures: string (nullable = true)
 |-- type_of_set_aside: string (nullable = true)
 |-- awarding_agency_code: string (nullable = true)
 |-- awarding_subagency_code: string (nullable = true)
 |-- funding_agency_name: string (nullable = true)
 |-- naics2: string (nullable = true)
 |-- psc2: string (nullable = true)
 |-- recipient_state: string (nullable = true)
 |-- recipient_country: string (nullable = true)
 |-- pop_state: string (nullable = true)
 |-- pop_country: string (nullable = true)
 |-- action_year: double (nullable = true)
 |-- action_month: double (nullable = true)
 |-- action_quarter: double (nullable = true)
 |-- durat

In [128]:
# Purpose: Inspect the Spark DataFrame schema before continuing with transformations.
offers_train_vector.printSchema()

root
 |-- id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- piid: string (nullable = true)
 |-- source_year: integer (nullable = true)
 |-- split: string (nullable = true)
 |-- offers_label: double (nullable = true)
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)



In [129]:
# Purpose: Import the PySpark DataFrame, ML, evaluation, plotting, and utility components used in this stage.
from pyspark.ml.regression import (LinearRegression, DecisionTreeRegressor, RandomForestRegressor, GBTRegressor)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

# persist vectorized datasets

In [130]:
# Purpose: Import the Python and PySpark utilities required for this notebook stage.
from pyspark import StorageLevel
offers_train_f = (offers_train_f.select("id", "award_id", "piid", "source_year", "split",
        "offers_label", "label",
        "features"
    )
    .persist(
        StorageLevel.DISK_ONLY
    )
)
offers_val_f = (
    offers_val_f
    .select(
        "id",
        "award_id",
        "piid",
        "source_year",
        "split",
        "offers_label",
        "label",
        "features"
    )
    .persist(
        StorageLevel.DISK_ONLY
    )
)
offers_test_f = (
    offers_test_f
    .select(
        "id",
        "award_id",
        "piid",
        "source_year",
        "split",
        "offers_label",
        "label",
        "features"
    )
    .persist(
        StorageLevel.DISK_ONLY
    )
)

In [131]:
# Purpose: Materialize the Spark DataFrame and verify the number of available records.
print("Train:", f"{offers_train_f.count():,}")
print("Validation:", f"{offers_val_f.count():,}")
print("Test:", f"{offers_test_f.count():,}"
)

Train: 5,128,990


Validation: 2,152,432


Test: 1,656,042


# Convert predictions back to original scale

In [125]:
# Purpose: Define `add_original_prediction` to convert model predictions back to the original target scale.
def add_original_prediction(predictions):
    return (predictions.withColumn("prediction_original", F.greatest(F.exp(F.col("prediction")) - F.lit(1.0),
                F.lit(0.0))
        )
    )

# Evaluation function

In [126]:
# Purpose: Define `evaluate_offers_model` to encapsulate the `evaluate_offers_model` processing logic for reuse.
def evaluate_offers_model(predictions):
    rmse = (RegressionEvaluator(labelCol="offers_label", predictionCol="prediction_original", metricName="rmse")
        .evaluate(predictions))
    mae = (RegressionEvaluator(labelCol="offers_label",
            predictionCol="prediction_original",
            metricName="mae"
        )
        .evaluate(predictions)
    )
    r2 = (
        RegressionEvaluator(
            labelCol="offers_label",
            predictionCol="prediction_original",
            metricName="r2"
        )
        .evaluate(predictions)
    )
    log_rmse = (
        RegressionEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="rmse"
        )
        .evaluate(predictions)
    )
    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Log_RMSE": log_rmse
    }

# MODEL 1 — LINEAR REGRESSION

In [127]:
# Purpose: Prepare `lr` for the next Big Data processing, analysis, or modeling step.
lr = LinearRegression(featuresCol="features", labelCol="label", predictionCol="prediction", maxIter=30,
    regParam=0.05, elasticNetParam=0.0, solver="l-bfgs")

In [128]:
# Purpose: Execute the code for the notebook section: MODEL 1 — LINEAR REGRESSION.
### Training the model

In [ ]:
# Purpose: Fit the configured Spark ML pipeline or regression model on the training data.
lr_model = (lr.fit(offers_train_f))

In [130]:
# Purpose: Write the processed Big Data or trained model artifact to the configured GCS location.
lr_model.write() \
    .overwrite() \
    .save(f"{MODEL_ROOT}/offers/linear_regression")

In [131]:
# Purpose: Import the PySpark DataFrame, ML, evaluation, plotting, and utility components used in this stage.
from pyspark.ml.regression import LinearRegressionModel
model_path = f"{MODEL_ROOT}/offers/linear_regression"
loaded_lr_model = LinearRegressionModel.load(model_path)
print("Model loaded successfully")
print("Coefficients size:", loaded_lr_model.coefficients.size)
print("Intercept:", loaded_lr_model.intercept)

Model loaded successfully
Coefficients size: 1480
Intercept: -54.23378897276648


In [ ]:
# Purpose: Prepare `rf` for the next Big Data processing, analysis, or modeling step.
rf = RandomForestRegressor(featuresCol="features", labelCol="label", predictionCol="prediction", numTrees=40,
    maxDepth=8, maxBins=32, minInstancesPerNode=10, featureSubsetStrategy="sqrt", subsamplingRate=0.8,
    seed=42
)